# RAG Evaluation

In [0]:
%pip install -r ../requirements.txt

In [0]:
dbutils.library.restartPython()

In [0]:
import pandas as pd
from langchain.chains import RetrievalQA
from langchain.document_loaders import WebBaseLoader
from langchain.text_splitter import CharacterTextSplitter
from langchain.vectorstores import Chroma
from langchain_openai import OpenAI, OpenAIEmbeddings

import mlflow

## 1.1-Create a simple function that runs each input through the RAG chain

In [0]:
def model(input_df):
  answer = []
  for index, row in input_df.iterrows():
      answer.append(qa(row["questions"]))

  return answer

## Load an eval dataset

In [0]:
#TODO: Load dataset

eval_df = pd.DataFrame(
    {
        "questions": [
            "What is MLflow?",
            "How to run mlflow.evaluate()?",
            "How to log_table()?",
            "How to load_table()?",
        ],
    }
)

## 1.2-Create a faithfulness metric

In [0]:
from mlflow.metrics.genai import EvaluationExample, faithfulness

# Create a good and bad example for faithfulness in the context of this problem
faithfulness_examples = [
  EvaluationExample(
      input="How do I disable MLflow autologging?",
      output="mlflow.autolog(disable=True) will disable autologging for all functions. In Databricks, autologging is enabled by default. ",
      score=2,
      justification="The output provides a working solution, using the mlflow.autolog() function that is provided in the context.",
      grading_context={
          "context": "mlflow.autolog(log_input_examples: bool = False, log_model_signatures: bool = True, log_models: bool = True, log_datasets: bool = True, disable: bool = False, exclusive: bool = False, disable_for_unsupported_versions: bool = False, silent: bool = False, extra_tags: Optional[Dict[str, str]] = None) → None[source] Enables (or disables) and configures autologging for all supported integrations. The parameters are passed to any autologging integrations that support them. See the tracking docs for a list of supported autologging integrations. Note that framework-specific configurations set at any point will take precedence over any configurations set by this function."
      },
  ),
  EvaluationExample(
      input="How do I disable MLflow autologging?",
      output="mlflow.autolog(disable=True) will disable autologging for all functions.",
      score=5,
      justification="The output provides a solution that is using the mlflow.autolog() function that is provided in the context.",
      grading_context={
          "context": "mlflow.autolog(log_input_examples: bool = False, log_model_signatures: bool = True, log_models: bool = True, log_datasets: bool = True, disable: bool = False, exclusive: bool = False, disable_for_unsupported_versions: bool = False, silent: bool = False, extra_tags: Optional[Dict[str, str]] = None) → None[source] Enables (or disables) and configures autologging for all supported integrations. The parameters are passed to any autologging integrations that support them. See the tracking docs for a list of supported autologging integrations. Note that framework-specific configurations set at any point will take precedence over any configurations set by this function."
      },
  ),
]

faithfulness_metric = faithfulness(model="openai:/gpt-4", examples=faithfulness_examples)
print(faithfulness_metric)

## 1.3-Create a relevance metric

In [0]:
from mlflow.metrics.genai import EvaluationExample, relevance

relevance_metric = relevance(model="openai:/gpt-4")
print(relevance_metric)

In [0]:
results = mlflow.evaluate(
  model,
  eval_df,
  model_type="question-answering",
  evaluators="default",
  predictions="result",
  extra_metrics=[faithfulness_metric, relevance_metric, mlflow.metrics.latency()],
  evaluator_config={
      "col_mapping": {
          "inputs": "questions",
          "context": "source_documents",
      }
  },
)
print(results.metrics)